<a href="https://colab.research.google.com/github/Sarthakpunj/MindSignal/blob/main/multimodal_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
#  MULTIMODAL DEPRESSION DETECTION — FULL PIPELINE v7 (fixed)
#  Fix: Cell 5 re-encodes video with libx264 + yuv420p so that
#       opencv can actually read frames (was silently returning
#       no frames due to codec incompatibility with -c copy)
# ================================================================


# ────────────────────────────────────────────────────────────────
# CELL 1: Install Dependencies
# ────────────────────────────────────────────────────────────────
# Run this cell, then RESTART RUNTIME, then continue from Cell 2.

!apt-get install -y cmake libopenblas-dev liblapack-dev -q
!pip install -q dlib
!pip install -q yt-dlp
!pip install -q librosa soundfile audioread
!pip install -q opencv-python-headless
!pip install -q transformers accelerate
!pip install -q torch torchaudio
!pip install -q matplotlib seaborn
!pip install -q pandas scikit-learn
!apt-get install -y ffmpeg -q

import subprocess
for pkg in ["numpy", "dlib"]:
    r = subprocess.run(["pip", "show", pkg], capture_output=True, text=True)
    for line in r.stdout.split("\n"):
        if line.startswith("Version"):
            print(f"  {pkg}: {line}")

print("\n✅ Done — RESTART RUNTIME now, then run from Cell 2")


# ────────────────────────────────────────────────────────────────
# CELL 2: Mount Drive + Set Paths
# ────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR   = "/content/drive/MyDrive/DepressionDetection/TestRun"
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(BASE_DIR,   exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Drive mounted → {BASE_DIR}")


# ────────────────────────────────────────────────────────────────
# CELL 3: Download Dlib Face Landmark Model
# ────────────────────────────────────────────────────────────────

import os, urllib.request, bz2

MODEL_PATH = "/content/shape_predictor_68_face_landmarks.dat"

if os.path.exists(MODEL_PATH):
    print(f"✅ Model already exists: {os.path.getsize(MODEL_PATH)/(1024*1024):.1f} MB")
else:
    print("⬇️  Downloading dlib 68-point landmark model (~100MB)...")
    URL = "https://github.com/davisking/dlib-models/raw/master/shape_predictor_68_face_landmarks.dat.bz2"
    bz2_path = MODEL_PATH + ".bz2"
    urllib.request.urlretrieve(URL, bz2_path)
    print("📦 Decompressing...")
    with bz2.open(bz2_path, 'rb') as f_in, open(MODEL_PATH, 'wb') as f_out:
        f_out.write(f_in.read())
    os.remove(bz2_path)
    print(f"✅ Model ready: {os.path.getsize(MODEL_PATH)/(1024*1024):.1f} MB")


# ────────────────────────────────────────────────────────────────
# CELL 4: Download Video
# ────────────────────────────────────────────────────────────────

import os
BASE_DIR  = "/content/drive/MyDrive/DepressionDetection/TestRun"
VIDEO_URL = "https://www.youtube.com/watch?v=F2hc2FLOdhI"

!yt-dlp -f "bestvideo[height<=480][ext=mp4]+bestaudio[ext=m4a]/mp4" \
    --merge-output-format mp4 \
    "{VIDEO_URL}" \
    -o "{BASE_DIR}/raw_video.mp4" \
    --no-playlist

!yt-dlp -x "{VIDEO_URL}" -o "{BASE_DIR}/raw_audio" --no-playlist

print("\n📂 Files saved:")
for f in sorted(os.listdir(BASE_DIR)):
    if f == "outputs": continue
    sz = os.path.getsize(os.path.join(BASE_DIR, f)) / (1024*1024)
    print(f"  {f}  ({sz:.1f} MB)")


# ────────────────────────────────────────────────────────────────
# CELL 5: Trim + Convert  ← THE FIX IS HERE
# ────────────────────────────────────────────────────────────────
# FIX: -c copy preserved the original YouTube codec (av1/vp9)
# which opencv cannot decode, so cap.read() silently returned
# nothing. Re-encoding with libx264 + yuv420p is universally
# readable by opencv on any platform.

import cv2

TRIM_SEC   = 300
VIDEO_PATH = os.path.join(BASE_DIR, "test_video_trimmed.mp4")
AUDIO_PATH = os.path.join(BASE_DIR, "test_audio_trimmed.wav")

RAW_VIDEO, RAW_AUDIO = None, None
for f in os.listdir(BASE_DIR):
    if "raw_video" in f and f.endswith(".mp4"):
        RAW_VIDEO = os.path.join(BASE_DIR, f)
    if "raw_audio" in f:
        for ext in [".wav", ".m4a", ".webm", ".opus", ".mp3"]:
            if f.endswith(ext):
                RAW_AUDIO = os.path.join(BASE_DIR, f)

print(f"Raw video : {RAW_VIDEO}")
print(f"Raw audio : {RAW_AUDIO}")

if RAW_VIDEO and RAW_AUDIO:
    # Re-encode to h264/yuv420p — opencv can always read this
    ret = os.system(
        f'ffmpeg -i "{RAW_VIDEO}" -t {TRIM_SEC} '
        f'-c:v libx264 -preset fast -crf 23 -pix_fmt yuv420p '
        f'-c:a aac -y "{VIDEO_PATH}" -loglevel error'
    )
    os.system(
        f'ffmpeg -i "{RAW_AUDIO}" -t {TRIM_SEC} '
        f'-ar 16000 -ac 1 -y "{AUDIO_PATH}" -loglevel error'
    )

    # Verify opencv can actually decode frames
    cap = cv2.VideoCapture(VIDEO_PATH)
    ok, frame = cap.read()
    cap.release()

    for path, label in [(VIDEO_PATH, "Video"), (AUDIO_PATH, "Audio")]:
        if os.path.exists(path) and os.path.getsize(path) > 1000:
            print(f"✅ {label}: {os.path.getsize(path)/(1024*1024):.1f} MB")
        else:
            print(f"❌ {label}: failed")

    if ok:
        print(f"✅ OpenCV read test passed — frame shape: {frame.shape}")
    else:
        print("❌ OpenCV still cannot read — check ffmpeg output above")
else:
    print("❌ Raw files not found — check Cell 4 output")


# ────────────────────────────────────────────────────────────────
# CELL 6: Imports + Path Setup
# ────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import torch
import librosa
import librosa.display
import cv2
import dlib
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings("ignore")

from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.decomposition import PCA
from tqdm import tqdm

BASE_DIR    = "/content/drive/MyDrive/DepressionDetection/TestRun"
OUTPUT_DIR  = os.path.join(BASE_DIR, "outputs")
VIDEO_PATH  = os.path.join(BASE_DIR, "test_video_trimmed.mp4")
AUDIO_PATH  = os.path.join(BASE_DIR, "test_audio_trimmed.wav")
MODEL_PATH  = "/content/shape_predictor_68_face_landmarks.dat"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# dlib 68-point landmark indices
RIGHT_EYE  = list(range(36, 42))
LEFT_EYE   = list(range(42, 48))
RIGHT_BROW = list(range(17, 22))
LEFT_BROW  = list(range(22, 27))
MOUTH_TOP, MOUTH_BOTTOM = 62, 66
MOUTH_LEFT, MOUTH_RIGHT = 60, 64

detector  = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(MODEL_PATH)

print(f"✅ dlib detector + predictor loaded")
print(f"✅ numpy  : {np.__version__}")
print(f"✅ Device : {DEVICE}")
print(f"✅ Video  : {os.path.exists(VIDEO_PATH)}")
print(f"✅ Audio  : {os.path.exists(AUDIO_PATH)}")

# Quick opencv sanity check
cap = cv2.VideoCapture(VIDEO_PATH)
ok, _frame = cap.read(); cap.release()
print(f"✅ OpenCV read: {'OK' if ok else 'FAILED — re-run Cell 5'}")


# ────────────────────────────────────────────────────────────────
# CELL 7: Load Audio + Visualize Waveform & Spectrogram
# ────────────────────────────────────────────────────────────────

print("🎵 Loading audio...")
y, sr    = librosa.load(AUDIO_PATH, sr=16000)
duration = len(y) / sr
print(f"  ✅ {duration:.1f}s | {sr}Hz | {len(y):,} samples | range [{y.min():.3f}, {y.max():.3f}]")

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(np.linspace(0, duration, len(y)), y,
             color='steelblue', alpha=0.7, linewidth=0.4)
axes[0].set_title("Waveform — Raw Audio Signal\nSpiky = active speech | Flat = pauses/silence")
axes[0].set_xlabel("Time (seconds)"); axes[0].set_ylabel("Amplitude")
axes[0].axhline(0, color='black', linewidth=0.5); axes[0].set_xlim(0, duration)

mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
mel_db   = librosa.power_to_db(mel_spec, ref=np.max)
img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel',
                                ax=axes[1], cmap='viridis')
plt.colorbar(img, ax=axes[1], label='dB')
axes[1].set_title("Mel Spectrogram\nBrighter = more energy | Depression = duller/lower")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "1_raw_audio.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved")


# ────────────────────────────────────────────────────────────────
# CELL 8: Handcrafted Audio Features (yin pitch — no numba)
# ────────────────────────────────────────────────────────────────

def extract_handcrafted_audio_features(y, sr):
    features, timeseries = {}, {}
    hop = 512

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=hop)
    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = float(np.mean(mfccs[i]))
        features[f"mfcc_{i+1}_std"]  = float(np.std(mfccs[i]))
    timeseries["mfccs"] = mfccs
    print("  ✅ MFCCs: 26 features")

    f0          = librosa.yin(y, fmin=librosa.note_to_hz('C2'),
                                fmax=librosa.note_to_hz('C7'), sr=sr, hop_length=hop)
    rms_full    = librosa.feature.rms(y=y, hop_length=hop)[0]
    min_len     = min(len(f0), len(rms_full))
    f0          = f0[:min_len];  rms_full = rms_full[:min_len]
    voiced_flag = rms_full > 0.01
    f0_voiced   = f0[voiced_flag]

    features["pitch_mean"]      = float(np.nanmean(f0_voiced)) if len(f0_voiced) > 0 else 0.0
    features["pitch_std"]       = float(np.nanstd(f0_voiced))  if len(f0_voiced) > 0 else 0.0
    features["pitch_range"]     = float(np.nanmax(f0_voiced) - np.nanmin(f0_voiced)) if len(f0_voiced) > 0 else 0.0
    features["voiced_fraction"] = float(np.mean(voiced_flag))
    timeseries["f0"] = f0;  timeseries["voiced_flag"] = voiced_flag
    print(f"  ✅ Pitch: mean={features['pitch_mean']:.1f}Hz, voiced={features['voiced_fraction']:.1%}")

    rms = rms_full
    features["energy_mean"]  = float(np.mean(rms))
    features["energy_std"]   = float(np.std(rms))
    features["energy_max"]   = float(np.max(rms))
    features["energy_min"]   = float(np.min(rms))
    features["energy_range"] = features["energy_max"] - features["energy_min"]
    timeseries["rms"] = rms
    print(f"  ✅ Energy: mean={features['energy_mean']:.4f}")

    is_silence  = rms < 0.01
    transitions = np.diff(is_silence.astype(int))
    features["pause_fraction"]  = float(np.mean(is_silence))
    features["num_pauses"]      = int(np.sum(transitions == 1))
    features["speech_fraction"] = float(1 - np.mean(is_silence))
    timeseries["is_silence"] = is_silence
    print(f"  ✅ Pauses: {features['num_pauses']} pauses, {features['pause_fraction']:.1%} silence")

    spec_centroid  = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
    spec_rolloff   = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop)[0]
    spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
    zcr            = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
    features["spectral_centroid_mean"]  = float(np.mean(spec_centroid))
    features["spectral_centroid_std"]   = float(np.std(spec_centroid))
    features["spectral_rolloff_mean"]   = float(np.mean(spec_rolloff))
    features["spectral_bandwidth_mean"] = float(np.mean(spec_bandwidth))
    features["zcr_mean"] = float(np.mean(zcr))
    features["zcr_std"]  = float(np.std(zcr))
    timeseries["spectral_centroid"] = spec_centroid
    print(f"  ✅ Spectral centroid: {features['spectral_centroid_mean']:.0f}Hz")

    chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=hop)
    features["chroma_mean"] = float(np.mean(chroma))
    features["chroma_std"]  = float(np.std(chroma))
    print(f"  ✅ Chroma | Total features: {len(features)}")
    return features, timeseries


print("📊 Extracting handcrafted audio features...")
audio_features, audio_ts = extract_handcrafted_audio_features(y, sr)
pd.DataFrame([audio_features]).to_csv(
    os.path.join(OUTPUT_DIR, "audio_handcrafted_features.csv"), index=False)
print("✅ Saved")


# ────────────────────────────────────────────────────────────────
# CELL 9: Plot Audio Features Over Time
# ────────────────────────────────────────────────────────────────

hop = 512
t   = np.arange(len(audio_ts["rms"])) * hop / sr

fig, axes = plt.subplots(4, 1, figsize=(14, 13))

f0_plot = audio_ts["f0"].copy().astype(float)
f0_plot[~audio_ts["voiced_flag"]] = np.nan
t_f0 = np.arange(len(f0_plot)) * hop / sr
axes[0].plot(t_f0, f0_plot, color='royalblue', linewidth=1.5, label='F0 (Pitch)')
axes[0].set_title("Pitch (F0) Over Time\n↓ Lower + less variable = monotone = depression indicator")
axes[0].set_ylabel("Hz"); axes[0].set_xlim(0, duration); axes[0].legend()

axes[1].plot(t, audio_ts["rms"], color='darkorange', linewidth=1, label='RMS Energy')
axes[1].fill_between(t, 0, audio_ts["rms"], where=audio_ts["is_silence"],
                     color='red', alpha=0.35, label='Silence/Pause')
axes[1].axhline(0.01, color='red', linestyle='--', alpha=0.5, linewidth=0.8, label='Silence threshold')
axes[1].set_title("Energy + Pauses\n↑ More red = more pauses = depression indicator")
axes[1].set_ylabel("RMS"); axes[1].set_xlim(0, duration); axes[1].legend()

t_sc = np.arange(len(audio_ts["spectral_centroid"])) * hop / sr
axes[2].plot(t_sc, audio_ts["spectral_centroid"], color='mediumseagreen', linewidth=1)
axes[2].axhline(np.mean(audio_ts["spectral_centroid"]), color='red', linestyle='--',
                linewidth=0.8, label=f"Mean: {np.mean(audio_ts['spectral_centroid']):.0f}Hz")
axes[2].set_title("Spectral Centroid\n↓ Lower = duller voice = depression indicator")
axes[2].set_ylabel("Hz"); axes[2].set_xlim(0, duration); axes[2].legend()

mfcc_img = axes[3].imshow(audio_ts["mfccs"], aspect='auto', origin='lower',
                           cmap='coolwarm', extent=[0, duration, 1, 13])
plt.colorbar(mfcc_img, ax=axes[3], label='Value')
axes[3].set_title("MFCCs Over Time\n↓ Less variation = monotone voice = depression indicator")
axes[3].set_ylabel("MFCC #"); axes[3].set_xlabel("Time (seconds)")

plt.suptitle("Audio Feature Analysis", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "2_audio_features_over_time.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved")


# ────────────────────────────────────────────────────────────────
# CELL 10: Wav2Vec2 Deep Audio Embeddings
# ────────────────────────────────────────────────────────────────

print("🤖 Loading Wav2Vec2 (~360MB)...")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model_w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)
model_w2v.eval()
print("✅ Wav2Vec2 ready\n")


def extract_wav2vec2_embeddings(y, sr, segment_sec=10, overlap_sec=2):
    seg_len = segment_sec * sr
    hop_len = (segment_sec - overlap_sec) * sr
    embeddings, timestamps = [], []
    starts = list(range(0, len(y) - seg_len + 1, hop_len))
    print(f"  {len(starts)} segments of {segment_sec}s (overlap={overlap_sec}s)...")
    for i, start in enumerate(starts):
        seg = y[start: start + seg_len].copy()
        seg = seg / (np.abs(seg).max() + 1e-8)
        inputs = processor(seg, sampling_rate=sr, return_tensors="pt",
                           padding=True).input_values.to(DEVICE)
        with torch.no_grad():
            emb = model_w2v(inputs).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        embeddings.append(emb)
        timestamps.append(start / sr)
        if (i + 1) % 5 == 0 or (i + 1) == len(starts):
            print(f"    [{i+1}/{len(starts)}] done")
    embeddings = np.array(embeddings)
    print(f"\n  ✅ Shape: {embeddings.shape}")
    return embeddings, timestamps


embeddings, seg_timestamps = extract_wav2vec2_embeddings(y, sr)
np.save(os.path.join(OUTPUT_DIR, "wav2vec2_embeddings.npy"), embeddings)
print("✅ Embeddings saved")


# ────────────────────────────────────────────────────────────────
# CELL 11: Visualize Wav2Vec2 Embeddings
# ────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(embeddings.T, aspect='auto', cmap='RdBu_r',
               extent=[seg_timestamps[0], seg_timestamps[-1], 0, 768])
axes[0].set_title(f"Wav2Vec2 Matrix: {embeddings.shape[0]}×{embeddings.shape[1]}\n"
                  "Each column = 1 segment | Each row = 1 learned feature")
axes[0].set_xlabel("Time (seconds)"); axes[0].set_ylabel("Embedding Dim")

pca    = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)
sc = axes[1].scatter(emb_2d[:, 0], emb_2d[:, 1], c=seg_timestamps, cmap='plasma', s=80, zorder=3)
plt.colorbar(sc, ax=axes[1], label='Time (seconds)')
axes[1].plot(emb_2d[:, 0], emb_2d[:, 1], 'k-', alpha=0.2, linewidth=0.8, zorder=2)
axes[1].annotate("▶ Start", emb_2d[0],  fontsize=9, color='green', fontweight='bold')
axes[1].annotate("■ End",   emb_2d[-1], fontsize=9, color='red',   fontweight='bold')
axes[1].set_title(f"PCA (768→2D) | Variance: {pca.explained_variance_ratio_.sum():.1%}\n"
                  "Each dot = 10s segment")
axes[1].set_xlabel("PCA 1"); axes[1].set_ylabel("PCA 2")

plt.suptitle("Wav2Vec2 Deep Audio Representations", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "3_wav2vec2_embeddings.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved")


# ────────────────────────────────────────────────────────────────
# CELL 12: Test dlib on Single Frame
# ────────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(VIDEO_PATH)
ret, test_frame = cap.read()
cap.release()

if not ret:
    print("❌ Cannot read video — re-run Cell 5 (libx264 re-encode)")
else:
    gray  = cv2.cvtColor(test_frame, cv2.COLOR_BGR2GRAY)
    faces = detector(gray, 1)
    print(f"✅ Frame read: {test_frame.shape[1]}×{test_frame.shape[0]}")
    print(f"✅ Faces detected: {len(faces)}")

    if len(faces) > 0:
        shape = predictor(gray, faces[0])
        pts   = np.array([[shape.part(i).x, shape.part(i).y] for i in range(68)])
        vis   = test_frame.copy()
        for (x, y_pt) in pts:
            cv2.circle(vis, (x, y_pt), 2, (0, 255, 0), -1)
        plt.figure(figsize=(6, 4))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        plt.title("68 dlib landmarks detected ✅"); plt.axis('off'); plt.show()
        print("   Ready to process full video — run Cell 13")
    else:
        plt.figure(figsize=(6, 4))
        plt.imshow(cv2.cvtColor(test_frame, cv2.COLOR_BGR2RGB))
        plt.title("No face on first frame — check image"); plt.axis('off'); plt.show()
        print("❌ No face detected on first frame")


# ────────────────────────────────────────────────────────────────
# CELL 13: Process Full Video with dlib
# ────────────────────────────────────────────────────────────────

def landmarks_to_array(shape):
    return np.array([[shape.part(i).x, shape.part(i).y] for i in range(68)], dtype=float)

def ear(pts, idx):
    p = pts[idx]
    A = np.linalg.norm(p[1] - p[5])
    B = np.linalg.norm(p[2] - p[4])
    C = np.linalg.norm(p[0] - p[3])
    return float((A + B) / (2.0 * C + 1e-6))

def brow_height(pts, brow_idx, eye_idx):
    by = np.mean(pts[brow_idx, 1])
    ey = np.mean(pts[eye_idx,  1])
    return float(ey - by)


def process_video(video_path, sample_every_n=15, max_frames=300):
    cap   = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"  Video: {total} frames | {fps:.1f} fps | {W}×{H}")
    print(f"  Sampling every {sample_every_n} frames (~{fps/sample_every_n:.1f} samples/sec)")

    all_data, annotated = [], []
    frame_idx, sampled  = 0, 0
    pbar = tqdm(total=min(max_frames, total // sample_every_n), desc="Frames")

    while sampled < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % sample_every_n == 0:
            gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = detector(gray, 0)

            if len(faces) > 0:
                shape = predictor(gray, faces[0])
                pts   = landmarks_to_array(shape)
                fd    = {}

                mouth_open = abs(pts[MOUTH_TOP, 1] - pts[MOUTH_BOTTOM, 1])
                mouth_wide = abs(pts[MOUTH_LEFT, 0] - pts[MOUTH_RIGHT, 0])
                fd["mouth_openness"] = float(mouth_open)
                fd["mouth_width"]    = float(mouth_wide)
                fd["mouth_ratio"]    = float(mouth_open / (mouth_wide + 1e-6))

                fd["left_ear"]  = ear(pts, LEFT_EYE)
                fd["right_ear"] = ear(pts, RIGHT_EYE)
                fd["mean_ear"]  = (fd["left_ear"] + fd["right_ear"]) / 2

                fd["left_brow_height"]  = brow_height(pts, LEFT_BROW,  LEFT_EYE)
                fd["right_brow_height"] = brow_height(pts, RIGHT_BROW, RIGHT_EYE)
                fd["mean_brow_height"]  = (fd["left_brow_height"] + fd["right_brow_height"]) / 2

                eye_mid_x = (pts[36, 0] + pts[45, 0]) / 2
                eye_mid_y = (pts[36, 1] + pts[45, 1]) / 2
                fd["head_pitch"] = float(pts[30, 1] - eye_mid_y)
                fd["head_yaw"]   = float(pts[30, 0] - eye_mid_x)

                fd["frame_idx"]     = frame_idx
                fd["timestamp_sec"] = frame_idx / fps
                all_data.append(fd)

                if len(annotated) < 5:
                    vis = frame.copy()
                    for (px, py) in pts.astype(int):
                        cv2.circle(vis, (px, py), 2, (0, 255, 0), -1)
                    annotated.append((frame_idx, vis))

            sampled += 1
            pbar.update(1)
        frame_idx += 1

    cap.release()
    pbar.close()

    face_df  = pd.DataFrame(all_data)
    detected = len(face_df)
    print(f"\n  ✅ Detected : {detected}/{sampled} ({detected/max(sampled,1):.1%})")
    print(f"  ⚠️  Missing  : {sampled-detected}/{sampled}")
    return face_df, annotated


print("👁️  Processing video with dlib...")
face_df, annotated_frames = process_video(VIDEO_PATH, sample_every_n=15, max_frames=300)

if len(face_df) > 0:
    face_df.to_csv(os.path.join(OUTPUT_DIR, "visual_features_per_frame.csv"), index=False)
    print(f"\n✅ {len(face_df)} frames × {len(face_df.columns)} features saved")
    print(face_df[["timestamp_sec", "mean_ear", "mouth_openness", "mean_brow_height"]].head())
else:
    print("\n❌ face_df empty — check Cell 12 output")


# ────────────────────────────────────────────────────────────────
# CELL 14: Show Sample Frames
# ────────────────────────────────────────────────────────────────

if annotated_frames:
    n    = len(annotated_frames)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    axes = [axes] if n == 1 else list(axes)
    for ax, (fidx, frame) in zip(axes, annotated_frames):
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {fidx}", fontsize=9); ax.axis('off')
    plt.suptitle("Sample Frames — 68 Landmarks (dlib) ✅", fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "4_sample_faces.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved")
else:
    print("⚠️  No annotated frames")


# ────────────────────────────────────────────────────────────────
# CELL 15: Plot Facial Features Over Time
# ────────────────────────────────────────────────────────────────

if len(face_df) == 0:
    print("❌ Cannot plot — face_df is empty")
else:
    t = face_df["timestamp_sec"]
    fig, axes = plt.subplots(4, 1, figsize=(14, 13))

    axes[0].plot(t, face_df["mean_ear"], color='steelblue', linewidth=1.2, label='Mean EAR')
    axes[0].fill_between(t, face_df["left_ear"], face_df["right_ear"],
                         alpha=0.2, color='steelblue', label='L/R range')
    axes[0].axhline(face_df["mean_ear"].mean(), color='red', linestyle='--', linewidth=0.8,
                    label=f'Avg: {face_df["mean_ear"].mean():.3f}')
    axes[0].set_title("Eye Aspect Ratio\nNormal ≈ 0.25–0.35 | ↓ Lower = fatigue/low affect")
    axes[0].set_ylabel("EAR"); axes[0].legend(fontsize=8)

    axes[1].plot(t, face_df["mouth_openness"], color='darkorange', linewidth=1)
    axes[1].axhline(face_df["mouth_openness"].mean(), color='red', linestyle='--',
                    linewidth=0.8, label=f'Avg: {face_df["mouth_openness"].mean():.1f}px')
    axes[1].set_title("Mouth Openness\n↓ Less movement = flat affect = depression indicator")
    axes[1].set_ylabel("px"); axes[1].legend(fontsize=8)

    axes[2].plot(t, face_df["mean_brow_height"], color='mediumseagreen', linewidth=1.2)
    axes[2].axhline(face_df["mean_brow_height"].mean(), color='red', linestyle='--',
                    linewidth=0.8, label=f'Avg: {face_df["mean_brow_height"].mean():.1f}px')
    axes[2].set_title("Brow Height\n↓ Lower = furrowing = depression indicator")
    axes[2].set_ylabel("px"); axes[2].legend(fontsize=8)

    axes[3].plot(t, face_df["head_pitch"], color='mediumpurple', linewidth=1, label='Pitch (nod)')
    axes[3].plot(t, face_df["head_yaw"],   color='tomato',       linewidth=1, label='Yaw (turn)')
    axes[3].axhline(0, color='black', linewidth=0.5, linestyle='--')
    axes[3].set_title("Head Pose\n↓ Negative pitch = head down = depression indicator")
    axes[3].set_ylabel("px"); axes[3].set_xlabel("Time (s)"); axes[3].legend(fontsize=8)

    plt.suptitle("Facial Feature Analysis (dlib 68-point)", fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "5_visual_features_over_time.png"), dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved")


# ────────────────────────────────────────────────────────────────
# CELL 16: Aggregate Visual Features + Final Summary
# ────────────────────────────────────────────────────────────────

if len(face_df) > 0:
    skip = ["frame_idx", "timestamp_sec"]
    agg  = {}
    for col in [c for c in face_df.columns if c not in skip]:
        v = face_df[col].dropna()
        if len(v) == 0: continue
        agg[f"{col}_mean"] = float(v.mean())
        agg[f"{col}_std"]  = float(v.std())
        agg[f"{col}_min"]  = float(v.min())
        agg[f"{col}_max"]  = float(v.max())

    with open(os.path.join(OUTPUT_DIR, "visual_agg_features.json"), "w") as f:
        json.dump(agg, f, indent=2)

    print("🔑 Key interview-level stats:")
    for key, label in [
        ("mean_ear_mean",         "Eye openness avg  "),
        ("mouth_openness_mean",   "Mouth openness avg"),
        ("mean_brow_height_mean", "Brow height avg   "),
        ("head_pitch_mean",       "Head pitch avg    "),
    ]:
        if key in agg:
            print(f"  {label}: {agg[key]:.3f}")
    print(f"\n✅ Aggregated to {len(agg)} interview-level features")

print("\n" + "="*55)
print("  ✅  PIPELINE COMPLETE")
print("="*55)
print(f"""
Outputs → {OUTPUT_DIR}

  Plots:
    1_raw_audio.png                → Waveform + Mel Spectrogram
    2_audio_features_over_time.png → Pitch, Energy, Pauses, MFCCs
    3_wav2vec2_embeddings.png      → Deep embedding heatmap + PCA
    4_sample_faces.png             → Sample frames with 68 landmarks
    5_visual_features_over_time.png → EAR, Mouth, Brow, Head pose

  Feature files:
    audio_handcrafted_features.csv → {len(audio_features)} features
    wav2vec2_embeddings.npy        → shape {embeddings.shape}
    visual_features_per_frame.csv  → {len(face_df)} frames
    visual_agg_features.json       → interview-level stats

When DAIC-WOZ arrives:
  → Replace VIDEO_PATH / AUDIO_PATH with real participant files
  → Loop over all participants using the same functions above
  → Feed extracted features into DepressionDetector model
""")